# AlphaZero — Mastering Chess and Shogi by Self-Play with a General RL Algorithm (2017)

_Papers · Reinforcement Learning_

**One network predicts a move policy and a win/lose value; Monte-Carlo Tree Search uses it to look ahead, and self-play teaches the network from the search — no human games, no hand-crafted rules.**

---

This notebook is a practice scaffold for the **AlphaZero — Mastering Chess and Shogi by Self-Play with a General RL Algorithm (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch (Colab)

We build a tiny AlphaZero for Tic-Tac-Toe and run the full self-play loop. We go in four steps: (1) sanity-check the two worked examples (PUCT selection and backup), (2) lay down the game rules and the `(p, v)` network, (3) build MCTS plus self-play and training, and (4) run the loop and test optimal play with and without the network.

### Step 1 — Sanity-check the two worked examples

Before any search, we reproduce the lesson's two hand-worked numbers. The **PUCT** formula $U = c_{puct}\,P\,\sqrt{\sum_b N}/(1+N)$ adds an exploration bonus to each move's value $Q$; the move with the largest $Q+U$ is chosen. The **backup** flips the leaf value's sign at each ply (a zero-sum, alternating game) and updates the edge's running average $Q=W/N$.

In [ ]:
# Toy AlphaZero on Tic-Tac-Toe.  torch is preinstalled in Colab.
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

# ---------- 0. Sanity-check the lesson's two worked examples. ----------
c_puct = 1.5
parent_visits = 9                       # sum_b N(s,b); sqrt = 3

def U(P, N):
    return c_puct * P * math.sqrt(parent_visits) / (1 + N)

U_A = U(0.6, 3)                         # 0.675
U_B = U(0.4, 1)                         # 0.90
score_A = 0.10 + U_A
score_B = 0.50 + U_B
descend = "B" if score_B > score_A else "A"
print("PUCT:  U_A =", round(U_A, 3), " score_A =", round(score_A, 3),
      "|  U_B =", round(U_B, 3), " score_B =", round(score_B, 3),
      "-> descend into", descend)

# backup with sign flip: leaf v=+0.8
N1, W1 = 1, 0.2
N1, W1 = N1 + 1, W1 + (-0.8)            # opponent edge: -0.8
N2, W2 = 3, 0.5
N2, W2 = N2 + 1, W2 + (+0.8)            # our edge (flip again): +0.8
print("backup: edge1 N,W,Q =", N1, round(W1, 2), round(W1 / N1, 3),
      "| edge2 N,W,Q =", N2, round(W2, 2), round(W2 / N2, 3))
# -> U_A=0.675 score_A=0.775 | U_B=0.9 score_B=1.4 -> descend into B
# -> backup edge1 Q=-0.3 | edge2 Q=0.325

### Step 2 — Tic-Tac-Toe rules and the `(p, v)` network

The board is 9 cells, always written from the side-to-move's frame (`+1` = the player about to move). `step` plays a move and negates the board so the next player is again `+1`. The network $f_\theta(s)$ outputs a move-policy `p` (masked to legal moves) and a scalar value `v` in $[-1, 1]$ via `tanh`.

In [ ]:
# ---------- 1. Tic-Tac-Toe rules.  board: 9 cells, +1 = player to move, -1 = other. ----------
LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def legal(b):
    return [i for i in range(9) if b[i] == 0]

def winner(b):                                          # +1 / -1 if a line is filled, else 0
    for a, c, d in LINES:
        if b[a] != 0 and b[a] == b[c] == b[d]:
            return b[a]
    return 0

def step(b, a):                                         # play 'a' for the side-to-move (+1), flip frame
    nb = b[:]
    nb[a] = 1
    return [-x for x in nb]                              # negate so the next player is again "+1"

def terminal(b):
    return winner(b) != 0 or len(legal(b)) == 0

# ---------- 2. Tiny (p, v) = f_theta(s) network. ----------
class Net(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(9, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())
        self.pi   = nn.Linear(h, 9)                     # policy logits
        self.v    = nn.Linear(h, 1)                     # value
    def forward(self, x):
        z = self.body(x)
        return self.pi(z), torch.tanh(self.v(z)).squeeze(-1)

def evaluate(net, b):                                   # masked policy + value for one board
    x = torch.tensor(b, dtype=torch.float32)
    with torch.no_grad():
        logits, v = net(x.unsqueeze(0))
    logits = logits.squeeze(0).clone()
    mask = torch.full((9,), float('-inf'))
    mask[legal(b)] = 0.0
    p = F.softmax(logits + mask, dim=0)                 # illegal moves -> prob 0
    return p.numpy(), float(v)

### Step 3 — MCTS, self-play, and the training loss

`mcts` runs simulations that **select** down the tree by PUCT, **expand+evaluate** a leaf (network value, or a random rollout in the ablation), and **back up** the value with a sign flip at each ply. `self_play` turns a finished game into `(state, π, z)` triples — `π` from visit counts, `z` the final result in each position's frame. `train` minimises the paper's loss: $(z-v)^2$ for value plus $-\pi^\top\log p$ for policy, with weight decay supplied by the optimizer.

In [ ]:
# ---------- 3. MCTS with per-edge {N,W,Q,P} and PUCT.  use_net=False -> ABLATION. ----------
class Node:
    def __init__(self):
        self.N = {}
        self.W = {}
        self.Q = {}
        self.P = {}
        self.kids = {}
        self.expanded = False

def rollout_value(b):                                   # ablation leaf value: random play to the end
    # Returns the outcome from the perspective of the side to move at b (+1 win, -1 loss, 0 draw).
    b = b[:]
    plies = 0
    while not terminal(b):
        b = step(b, random.choice(legal(b)))
        plies += 1
    if winner(b) == 0:
        return 0.0                                         # draw
    # The LAST mover (the winner) was the original side to move iff the number of
    # plies played from b is odd (ply 1 = original mover, ply 2 = opponent, ...).
    return 1.0 if (plies % 2 == 1) else -1.0

def mcts(net, root_board, sims=50, use_net=True):
    root = Node()
    nodes = {tuple(root_board): root}
    def expand(node, b):
        if use_net:
            p, v = evaluate(net, b)
        else:
            p = [0.0] * 9
            for a in legal(b):
                p[a] = 1.0 / len(legal(b))               # uniform prior
            v = rollout_value(b)                          # value from the side-to-move's view
        for a in legal(b):
            node.N[a] = 0
            node.W[a] = 0.0
            node.Q[a] = 0.0
            node.P[a] = p[a]
        node.expanded = True
        return v
    for _ in range(sims):
        b = root_board[:]
        node = root
        path = []
        # ---- SELECT down to a leaf by PUCT ----
        while node.expanded and not terminal(b):
            tot = sum(node.N.values())
            best, ba = -1e9, None
            for a in legal(b):
                u = c_puct * node.P[a] * math.sqrt(tot + 1e-8) / (1 + node.N[a])
                s = node.Q[a] + u
                if s > best:
                    best, ba = s, a
            path.append((node, ba))
            b = step(b, ba)
            key = tuple(b)
            node = nodes.setdefault(key, Node())
        # ---- EXPAND + EVALUATE leaf ----
        if terminal(b):
            # A finished line in winner(b)!=0 means the PREVIOUS mover won, so the
            # side to move at b has lost -> value -1 from its frame; draw -> 0.
            v = -1.0 if winner(b) != 0 else 0.0
        else:
            v = expand(node, b)
        # ---- BACKUP with sign flip ----
        for nd, a in reversed(path):
            v = -v                                        # flip: alternate players
            nd.N[a] += 1
            nd.W[a] += v
            nd.Q[a] = nd.W[a] / nd.N[a]
    counts = [root.N.get(a, 0) for a in range(9)]
    return counts

def policy_from_counts(counts, tau=1.0):
    c = [x ** (1.0 / tau) for x in counts]
    s = sum(c)
    return [x / s if s > 0 else 0.0 for x in c]

# ---------- 4. Self-play -> (s, pi, z) data. ----------
# Frame is normalized: every stored board is from the side-to-move's view (+1 = me).
# We record each (board, pi) and which ply it was, then label z by the final result
# relative to the player who was to move at that board.
def self_play(net, sims=50):
    b = [0] * 9
    data = []                                            # data = list of (board, pi)
    while not terminal(b):
        counts = mcts(net, b, sims=sims, use_net=True)
        pi = policy_from_counts(counts, tau=1.0)
        data.append((b[:], pi))                           # side to move here is "+1"
        a = random.choices(range(9), weights=pi)[0]
        b = step(b, a)
    # After the final step() the just-moved player's marks are -1, so winner(b) == -1
    # means the player who made the LAST move won; 0 == draw.
    last_mover_won = (winner(b) == -1)
    n = len(data)
    # The player to move at the LAST stored position made the final (winning) move.
    # So that position's z = +1 if last_mover_won; players alternate, so signs alternate
    # back through the game.
    zs = [0.0] * n
    if last_mover_won:
        for i in range(n):
            zs[i] = 1.0 if ((n - 1 - i) % 2 == 0) else -1.0
    # draw -> all zeros
    return [(s, pi, z) for (s, pi), z in zip(data, zs)]

# ---------- 5. Train on (s, pi, z) with Eq. (1). ----------
def train(net, opt, buf, epochs=5, bs=64):
    if not buf:
        return
    for _ in range(epochs):
        random.shuffle(buf)
        for i in range(0, len(buf), bs):
            batch = buf[i:i + bs]
            X  = torch.tensor([s for s, _, _ in batch], dtype=torch.float32)
            PI = torch.tensor([p for _, p, _ in batch], dtype=torch.float32)
            Z  = torch.tensor([z for _, _, z in batch], dtype=torch.float32)
            logits, v = net(X)
            logp = F.log_softmax(logits, dim=1)
            value_loss  = (Z - v).pow(2).mean()                 # (z - v)^2
            policy_loss = -(PI * logp).sum(dim=1).mean()        # -pi^T log p
            loss = value_loss + policy_loss                     # + weight decay via optimizer
            opt.zero_grad()
            loss.backward()
            opt.step()

### Step 4 — Run the AlphaZero loop and test optimal play

We iterate self-play → train for 12 rounds, keeping a rolling replay buffer. Then we test on two tactical positions (take the win, block the loss) where the optimal move is cell 2. The **guided** search (network priors + value) should solve both at a tiny budget; the **ablation** (uniform priors + random rollout) misses more often — isolating the network's contribution per simulation.

In [ ]:
# ---------- 6. Run the AlphaZero loop, then test optimal play. ----------
net = Net()
opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)  # weight_decay = Eq.1 c||theta||^2
buf = []
for it in range(12):
    for _ in range(20):
        buf += self_play(net, sims=40)
    buf = buf[-3000:]
    train(net, opt, buf, epochs=5)

def best_move(net, b, sims=80, use_net=True):
    return int(max(range(9), key=lambda a: (mcts(net, b, sims=sims, use_net=use_net)[a]) if a in legal(b) else -1))

# Test positions (board from side-to-move's frame, +1 = me):
take_win  = [1,1,0, 0,-1,0, 0,-1,0]   # I (+1) can complete top row at cell 2
block     = [-1,-1,0, 0,1,0, 0,1,0]   # opponent threatens top row; I must block cell 2
print("\n--- guided MCTS (network) ---")
print("take the win  -> picks cell", best_move(net, take_win,  use_net=True),  "(want 2)")
print("block to draw -> picks cell", best_move(net, block,     use_net=True),  "(want 2)")
print("--- ablation: MCTS WITHOUT the network (uniform prior + random rollout) ---")
print("take the win  -> picks cell", best_move(net, take_win,  use_net=False), "(want 2)")
print("block to draw -> picks cell", best_move(net, block,     use_net=False), "(want 2)")
# Guided MCTS reliably picks cell 2 in both at this tiny budget; the unguided
# search misses more often. (Our small run, not the paper's numbers.)

## Visualize the data & results

_Does the trained network make MCTS stronger PER SIMULATION? We take the same toy AlphaZero MCTS and, at each simulation budget, run it two ways on a held-out set of tactical Tic-Tac-Toe positions: GUIDED (network priors + network leaf value) vs UNGUIDED ablation (uniform priors + random rollout). We plot the fraction of positions where the search picks the optimal move._

In [ ]:
# Sketch of how the two curves above are produced (full MCTS in the CODE cell).
# Take the trained net. For each simulation budget, run MCTS on a fixed set of
# tactical positions (take-the-win / block-the-loss) two ways and record the
# fraction where the most-visited move equals the known-optimal move:
#
#   guided   = [solved_rate(mcts(net, pos, sims=s, use_net=True))  for s in budgets]
#   unguided = [solved_rate(mcts(net, pos, sims=s, use_net=False)) for s in budgets]
#
# use_net=True  -> P = network policy, leaf value = network v   (green, climbs fast)
# use_net=False -> P = uniform,        leaf value = random rollout (red, climbs slowly)
#
# The gap at small budgets is the network's contribution: it concentrates the few
# simulations on the right moves and gives less-noisy leaf values.
# (Numbers are from our own run; the paper reports chess/shogi/Go match results,
#  not these Tic-Tac-Toe rates.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked PUCT selection. With $c_{puct}=1.5$ and parent visits $\sum_b N(s,b)=9$ (so
            $\sqrt{9}=3$), Move A has $P=0.6,\,N=3,\,Q=0.10$ and Move B has $P=0.4,\,N=1,\,Q=0.50$. Compute
            each move's PUCT score and say which the simulation descends into.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Move A bonus: $U_A = 1.5\times 0.6\times \frac{3}{1+3} = 1.5\times0.6\times0.75 = 0.675$. — _$U = c_{puct}P\sqrt{\sum N}/(1+N)$; the $1+N=4$ in the denominator shrinks A's bonus because it was tried more._
- Move A score: $Q_A + U_A = 0.10 + 0.675 = 0.775$. — _PUCT picks $\arg\max(Q+U)$; A's value so far is mediocre._
- Move B bonus: $U_B = 1.5\times 0.4\times \frac{3}{1+1} = 1.5\times0.4\times1.5 = 0.90$. — _B has fewer visits ($1+N=2$), so a bigger bonus despite a smaller prior._
- Move B score: $Q_B + U_B = 0.50 + 0.90 = 1.40$. — _B both looked good early ($Q=0.5$) and is under-explored._

**Answer:** Scores: A $= 0.775$, B $= 1.40$. The simulation descends into Move B ($1.40\gt0.775$).
                 Even with the lower prior, B's strong early value plus its larger exploration bonus win this
                 step &mdash; PUCT balancing exploit against explore. The notebook recomputes these.

</details>

**Problem 2.** The worked backup. A simulation reaches a leaf the network values at $v=+0.8$ (for the player
            to move there). The edge one ply up (the opponent's move into the leaf) had $N=1,\,W=0.2$. Update
            that edge's $N,\,W,\,Q$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Flip the sign for this ply: the leaf worth $+0.8$ to its mover is worth $-0.8$ to the opponent who moved into it. — _Alternating zero-sum game: keep each $Q$ in its own player's frame._
- Increment the visit count: $N = 1 + 1 = 2$. — _One more simulation passed through this edge._
- Add the (flipped) value: $W = 0.2 + (-0.8) = -0.6$. — _$W$ accumulates the backed-up values for this edge._
- Recompute the mean: $Q = W/N = -0.6/2 = -0.30$. — _$Q$ is the running average outcome through this move._

**Answer:** After backup the edge has $N=2,\,W=-0.6,\,Q=-0.30$. (One ply further up, the sign flips
                 again to $+0.8$.) Forgetting the flip would have stored $+0.8$ here and made the opponent's
                 search optimize for our win &mdash; the classic bug. The notebook recomputes
                 $N=2,\,W=-0.6,\,Q=-0.30$.

</details>

**Problem 3.** The ablation: does the network actually help the search? Take your trained toy AlphaZero and
            run its MCTS two ways at the same simulation budget: (i) guided &mdash; network priors
            $P=\mathbf{p}$ and leaf value $v$ from the net; (ii) unguided &mdash; uniform priors
            and a random rollout to game end for the leaf value. Compare how well each plays the fixed
            test positions. What do you expect, and what does the gap show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the MCTS prior and leaf-value source; keep the simulation count, tree code, and test positions identical. — _An honest ablation changes exactly one thing — the network's guidance — so any difference is attributable to it._
- Run both on the test set: take-the-win, block-the-loss, and "play a full game, never lose". The guided search solves them at the tiny budget; the unguided one misses some. — _With only ~50 simulations, uniform priors spread visits thinly and random rollouts give noisy leaf values, so the search explores worse moves and evaluates them less reliably._
- Conclude that at a fixed, small budget the network's prior + value are what make the search strong. — _Both runs share the tree and budget; only the guided one plays optimally, isolating the network's guidance as the cause._

**Answer:** The guided MCTS plays optimally on the test positions at the small budget; the
                 unguided MCTS (uniform prior + random rollout) is weaker &mdash; it wastes its few
                 simulations on poor moves and gets noisy leaf values. Since the only difference is the network's
                 guidance, this isolates the prior $P$ and value $v$ as what makes the search strong per
                 simulation &mdash; the paper's whole point about searching 80k positions/s yet beating
                 engines that search 70M. The CODEVIZ panel shows this gap.

</details>